# v2: fine-tune v1 so it lifts its paws

Phase 4 found v1 skips small paw lifts: of 14 test motions that lift a paw, it did a median **14%** of
the lift (plain replay: 73%), and 12 were mostly skipped. The course's reward barely notices: the
body-position term averages 9 bodies with a 10 cm tolerance, so a 5 cm paw left down costs ~3% of one
term, and the verdicts only see the torso.

v2 is v1 **plus one reward term**, `paw_height` (`kflib/rewards.py`): the height of each of the four
paws against the reference's, `exp(-mean squared height error / PAW_STD²)`, weight `PAW_WEIGHT`.
With the defaults (2 cm, weight 1.0), a paw left on the ground during a 5 cm lift scores 0.21 instead
of 1. Everything else is v1: same library, same evaluation set, and **it starts from v1's weights**
(a fine-tune), so a few thousand iterations should be enough.

**Check after training:** on the evaluation set, HOLDS UP shouldn't drop. Then rerun Phase 4 with
`TRACKERS = "pgt_v1,pgt_v2"` to compare the paw lifts head to head (`foot_tracking`).

Needs a **G4 or A100 runtime**. **Nothing here moves the robot.** Tell the TA about the reward change
before anything trained with it goes near the robot.

In [ ]:
# @title Setup: pupper-mjlab (pinned), kflib, Drive (about 2 minutes)
import os, sys, subprocess
COMMIT = "aa475e9a393a209b29fd5de7e0ee10f3de643230"
BRANCH = "final-project-kflib"  # @param {type:"string"}
DATASET_NAME = "v1"  # @param {type:"string"}
RUN_NAME = "pgt_v2"  # @param {type:"string"}
INIT_FROM = "pgt_v1"  # @param {type:"string"}
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: switch to a G4 runtime")

if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q https://github.com/cs123-stanford/pupper-mjlab /content/pupper-mjlab
!git -C /content/pupper-mjlab checkout -q {COMMIT}
!pip install -q -e /content/pupper-mjlab
sys.path.insert(0, "/content/pupper-mjlab/src")
if not os.path.exists("/content/gemini_pupper"):
  !git clone -q -b {BRANCH} https://github.com/MlsBli/gemini_pupper /content/gemini_pupper
!git -C /content/gemini_pupper pull -q
sys.path.insert(0, "/content/gemini_pupper/final_project")

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pupper_general_tracker")
DATASET = ROOT / "datasets" / DATASET_NAME
RUNS = ROOT / "runs"
EVAL_DIR = RUNS / f"eval_{RUN_NAME}"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
from kflib.dataset import Dataset
ds = Dataset(DATASET)
print(len(ds.rows()), "motions in", DATASET)

In [ ]:
# @title Log into W&B (key from Colab Secrets: WANDB_API_KEY)
import wandb
try:
  from google.colab import userdata
  os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:  # noqa: BLE001
  pass
wandb.login()
ENTITY = wandb.Api().default_entity
print("W&B entity:", ENTITY)

In [ ]:
# @title The training library: v1's, reused
import json
LIB_NAME = f"{INIT_FROM}_train"
LIB_JSON = ROOT / "libraries" / f"{LIB_NAME}.json"
from kflib.runs import library_ids
assert json.loads(LIB_JSON.read_text())["ids"] == library_ids(ds), "the dataset changed since v1's library was built"
MOTION_ARTIFACT = f"{ENTITY}/mjlab/{LIB_NAME}"
wandb.Api().artifact(MOTION_ARTIFACT + ":latest")  # fails here if it isn't on W&B
print("library:", MOTION_ARTIFACT)

In [ ]:
# @title The fixed evaluation set, and its plain-replay scores (once; ~15 minutes the first time)
from kflib.runs import eval_set
from kflib.score import score
import shutil
for f in ("eval_set.json", "plain.csv"):  # the same evaluation set and plain-replay scores as v1
  if not (EVAL_DIR / f).exists():
    shutil.copyfile(RUNS / f"eval_{INIT_FROM}" / f, EVAL_DIR / f)
EVAL_JSON = EVAL_DIR / "eval_set.json"
if EVAL_JSON.exists():
  EVAL = json.loads(EVAL_JSON.read_text())
else:
  EVAL = [dict(r, npz=str(ds.motions / f"{r['id']}.npz")) for r in eval_set(ds)]
  EVAL_JSON.write_text(json.dumps(EVAL, indent=1))
from collections import Counter
print(len(EVAL), "motions:", dict(Counter((r["split"], r["source"]) for r in EVAL)))
WORKERS = min(8, os.cpu_count())
PLAIN = EVAL_DIR / "plain.csv"
if not PLAIN.exists():
  score(EVAL, None, out_csv=PLAIN, workers=WORKERS)
print("plain replay scores:", PLAIN)

## Fine-tune, in chunks

Each run of this cell trains up to `CHUNK` more iterations (never past `TARGET`), continuing from the
newest checkpoint of `RUN_NAME`; the very first chunk starts from `INIT_FROM`'s newest checkpoint
(iterations keep counting from there: v2's checkpoints are model_5000.pt, model_6000.pt, ... model_9999.pt). A 5,000-iteration chunk should take roughly 25-35 min if v1 runs at the
smoke test's speed (3,000 in ~15 min): check the iterations per second in the log and adjust.
Watch `Metrics/motion/error_*` and `Episode_Reward/motion_*` on W&B. Every checkpoint also writes a
`policy.json` with the whole library inside (~60 MB): that's the course's trick runner, harmless, and
not the file to deploy.

In [ ]:
# @title Train (continues from the newest checkpoint)
TARGET = 10000  # @param {type:"integer"}
PAW_STD = 0.02  # @param {type:"number"}
PAW_WEIGHT = 1.0  # @param {type:"number"}
CHUNK = 5000  # @param {type:"integer"}
import re
from kflib.runs import chunk_plan, latest_checkpoint
last, ckpt = latest_checkpoint(RUNS, RUN_NAME)  # model_4999.pt = 5,000 iterations done (rsl_rl counts from 0)
if ckpt is None:  # v2's first chunk: start from v1's weights
  last, ckpt = latest_checkpoint(RUNS, INIT_FROM)
  print(f"starting from {INIT_FROM}: {ckpt.parent.name}/{ckpt.name}")
todo, end = chunk_plan(last, ckpt is not None, TARGET, CHUNK)
print(f"{RUN_NAME}: " + (f"{last + 1} iterations done ({ckpt.parent.name}/{ckpt.name})" if ckpt else "starting")
      + (f", training up to {end + 1}" if todo else ": TARGET reached (raise it to keep going)"))
if todo > 0:
  os.chdir("/content/pupper-mjlab")
  from mjlab.scripts.train import TrainConfig, launch_training
  from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
  import mjlab.tasks  # registers the tasks
  TASK = "Mjlab-Tracking-Flat-Pupper-v3"
  env_cfg = load_env_cfg(TASK)
  env_cfg.scene.num_envs = 4096
  from mjlab.managers.reward_manager import RewardTermCfg
  from kflib.rewards import paw_height_error_exp
  env_cfg.rewards["paw_height"] = RewardTermCfg(
    func=paw_height_error_exp, weight=PAW_WEIGHT, params={"command_name": "motion", "std": PAW_STD})
  print("reward terms:", {k: v.weight for k, v in env_cfg.rewards.items()})
  agent_cfg = load_rl_cfg(TASK)
  agent_cfg.max_iterations = todo  # learn() runs this many MORE iterations after a resume
  agent_cfg.save_interval = 1000
  agent_cfg.logger = "wandb"
  agent_cfg.run_name = RUN_NAME
  if ckpt:  # resume from exactly the newest checkpoint (any session's folder)
    agent_cfg.resume = True
    agent_cfg.load_run = re.escape(ckpt.parent.name)
    agent_cfg.load_checkpoint = re.escape(ckpt.name)
  launch_training(TASK, TrainConfig(env=env_cfg, agent=agent_cfg, registry_name=MOTION_ARTIFACT, log_root=str(RUNS)))
  print("now:", latest_checkpoint(RUNS, RUN_NAME)[0] + 1, "iterations done")

## Score the newest checkpoint

Plays all evaluation motions with the policy (~15 min), next to plain replay. Scores are kept per
checkpoint (`eval_<iteration>.csv`), and the last cell compares them all, so you can see whether more
training still helps.

In [ ]:
# @title Score the newest checkpoint on the evaluation set
import csv
from kflib.runs import latest_checkpoint, summarize
from kflib.score import score
it, ckpt = latest_checkpoint(RUNS, RUN_NAME)
out = EVAL_DIR / f"eval_{it}.csv"
if out.exists():
  print("already scored:", out)
else:
  score(EVAL, ckpt, out_csv=out, workers=WORKERS, plain=False)
rows = list(csv.DictReader(out.open()))
for split, s in summarize(rows).items():
  print(f"\n{split}: {s['holds_up']}/{s['n']} HOLDS UP, {s['wobbles']} WOBBLES, {s['falls']} FALLS"
        f" | median {s['median_cm']} cm {s['median_deg']} deg | worst {s['worst_cm']} cm {s['worst_deg']} deg")
  for w in s["worst"]:
    print("   ", w)
plain = summarize([dict(r, mode="policy") for r in csv.DictReader(PLAIN.open())])
print("\nplain replay, for comparison:", {k: f"{v['holds_up']}/{v['n']} HOLDS UP, median {v['median_cm']} cm {v['median_deg']} deg" for k, v in plain.items()})

In [ ]:
# @title Progress across checkpoints
import csv, re
from kflib.runs import summarize
for f in sorted(EVAL_DIR.glob("eval_*.csv"), key=lambda p: int(re.findall(r"\d+", p.stem)[0])):
  s = summarize(list(csv.DictReader(f.open())))
  print(f"{f.stem:12}", " | ".join(f"{k}: {v['holds_up']}/{v['n']} HOLDS UP, median {v['median_cm']} cm {v['median_deg']} deg" for k, v in s.items()))

In [ ]:
# @title (Optional) Watch the policy on one motion (open the share link; stop the cell when done)
from kflib.runs import latest_checkpoint
WATCH = EVAL[0]["id"]  # @param {type:"string"}
it, ckpt = latest_checkpoint(RUNS, RUN_NAME)
!python -m mjlab.tasks.tracking.config.pupper.play_motion {ds.motions / (WATCH + '.npz')} --checkpoint {ckpt} --viewer viser --share True --loops 2